# MNIST Model Inference with Koopman

In [ ]:
from ast import literal_eval

import einops
import matplotlib.pyplot as plt
import matrepr
import numpy as np
import numpy.random as rnd
import pykoopman as pk
import torch
from matrepr import mprint
from pydmd import DMD, FbDMD
from safetensors.torch import load_file, load_model, save_model
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
import warnings
from torcheval.metrics import MulticlassAccuracy
from torchvision import datasets, transforms
import seaborn as sns
from learningdynamics.dmd_data import reshape_append_time
from learningdynamics.models import MLP, init_weights
from learningdynamics.utils import (
    compute_model_accuracy,
    getActivation,
    safetensors_metadata_parser,
)
from learningdynamics.koopman_inference import inference_with_koopman

In [ ]:
# | hide
np.random.seed(42)  # for reproducibility
%matplotlib inline

In [ ]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),  # first, convert image to PyTorch tensor
        transforms.Normalize((0.1307,), (0.3081,)),  # normalize inputs
    ]
)
train_set = datasets.MNIST(
    root="/mnt/datasets/", train=True, download=True, transform=transform
)
test_set = datasets.MNIST(
    root="/mnt/datasets/", train=False, download=True, transform=transform
)

In [ ]:
LAYER_START = 1

In [ ]:
# Load states
file_path = f"../saved_weights/mnist_start{LAYER_START}_states.safetensors"
metadata = safetensors_metadata_parser(file_path=file_path)
NUM_TRAJECTORIES = literal_eval(metadata["num_trajectories"])
NUM_STATES = literal_eval(metadata["num_states"])
NUM_ITERATIONS = literal_eval(metadata["num_iterations"])
NUM_STATES_OF_INTEREST = literal_eval(metadata["num_usable_states"])
data = load_file(file_path)

In [ ]:
# Load model
file_path = f"../saved_weights/mnist_start{LAYER_START}_scaled_model.safetensors"
metadata = safetensors_metadata_parser(file_path=file_path)
scaled_model = MLP(
    config=literal_eval(metadata["config"]),
    nonlinearity=metadata["nonlinearity"],
    in_features=784,
    out_features=10,
)
_ = load_model(scaled_model, file_path, device="cuda:0")

In [ ]:
t = np.arange(0, NUM_ITERATIONS, 1)

In [ ]:
# Example values for NUM_TRAJECTORIES and NUM_DELAYS
NUM_TRAJECTORIES_LIST = [1, 10, 50, 500]
NUM_DELAYS_LIST = list(range(1,11))
TEST_SAMPLES = 10_000

# Dictionary to store the eigenvalues for each combination of trajectories and delays
accuracy_dict = {}

for num_trajectories in NUM_TRAJECTORIES_LIST:
    for num_delays in NUM_DELAYS_LIST:
        print(f"h={num_delays},r={num_trajectories}...")
        train_states = reshape_append_time(
            data["train_states"][:NUM_ITERATIONS, :num_trajectories, :NUM_STATES]
        )
        observables = pk.observables.TimeDelay(delay=1, n_delays=num_delays)

        regressor = pk.regression.EDMD(svd_rank=500)
        dmd_model = pk.Koopman(observables=observables, regressor=regressor)
        _ = dmd_model.fit(train_states.numpy().T)

        _, accuracy =  inference_with_koopman(
            nn_model=scaled_model,
            dmd_model=dmd_model,
            test_set=test_set,
            layer_start=literal_eval(metadata["layer_start"]),
            num_iterations=NUM_ITERATIONS,
            num_delays=num_delays,
            num_usable_states=NUM_STATES_OF_INTEREST,
            test_size=TEST_SAMPLES,
        )
        accuracy_dict[(num_trajectories, num_delays)] = accuracy

h=1,r=1...


h=2,r=1...
h=3,r=1...
h=4,r=1...
h=5,r=1...
h=6,r=1...
h=7,r=1...
h=8,r=1...
h=9,r=1...
h=10,r=1...
h=1,r=10...
h=2,r=10...
h=3,r=10...
h=4,r=10...
h=5,r=10...
h=6,r=10...
h=7,r=10...
h=8,r=10...
h=9,r=10...
h=10,r=10...
h=1,r=50...
h=2,r=50...
h=3,r=50...
h=4,r=50...
h=5,r=50...
h=6,r=50...
h=7,r=50...
h=8,r=50...
h=9,r=50...
h=10,r=50...
h=1,r=500...
h=2,r=500...
h=3,r=500...
h=4,r=500...
h=5,r=500...
h=6,r=500...
h=7,r=500...
h=8,r=500...
h=9,r=500...
h=10,r=500...
